## ML Classical Baselines

In [1]:
import pandas as pd
from IPython.display import display

from utils.config import (
    BANDS_TO_RUN,
    DATA_DIR,
    DEFAULT_OVERLAP_SIZE,
    DEFAULT_WINDOW_SIZE,
)
from utils.ML.ml_pipeline import (
    best_confusion_predictions,
    get_cache_path,
    get_results_path,
    load_all_predictions,
    load_feature_dataframes,
    load_lovo_summary_tables,
    load_or_build_none_reference_features,
    load_params_lookup,
    load_raw_csi_data,
    lovo_aggregated_analysis_table,
    master_results_table,
    per_room_position_accuracy_table,
    print_normalization_discriminability,
    process_magnitude_data,
    run_global_baselines,
    run_optional_grid_search,
    save_analysis_tables,
    save_lovo_analysis_table,
)
from utils.plots import (
    plot_band_error_cdf,
    plot_block_vs_lovo_position_accuracy,
    plot_floor_plan_heatmap,
    plot_global_position_confusion_matrix,
    plot_localization_error_cdf_by_model,
    plot_lovo_fold_spread,
    plot_magnitude_analysis_interactive,
    plot_model_band_error_boxplot,
    plot_position_confusion_by_true_room,
)

#### Project Configurations

In [2]:
CALIBRATION_MODE = "rssi"   # ("none", "packet_norm", "rssi")

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
    "calibration_eps": 1e-12,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline",  # none | zscore | minmax | packet_minmax | empty_baseline
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_session"  # per_session | per_user | global (empty_baseline only)
SHOW_MAGNITUDE_PLOT = False   # True -> interactively plot raw vs normalized CSI

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": DEFAULT_WINDOW_SIZE,
    "overlap_size": DEFAULT_OVERLAP_SIZE,
    "require_all_esps": False,
}

MODELS_TO_RUN = ("RF", "KNN", "SVM")
SPLIT_MODES = ("lovo",)  # cross_session, lovo, random, block
RUN_GRID_SEARCH = False
REQUIRE_TUNED_PARAMS = True  # True -> require an exact matching grid-search run
FORCE_RETRAIN = True
SAVE_PREDICTIONS = True
N_JOBS = 8

BLOCK_COUNT = 10
TEST_SIZE = 0.30
RANDOM_STATE = 42
ROW_SPACING = 1.0
COLUMN_SPACING = 1.0
SVM_FUSION_FALLBACK_SECONDS = 30 * 60

CONFUSION_DATASET = "Fusion"
CONFUSION_MODEL = "best"      # "best", "RF", "KNN", or "SVM"

SHOW_CDF_BY_BAND = True
SHOW_CDF_BY_MODEL = True
SHOW_BOXPLOT = True
SHOW_FLOOR_PLAN = True
SHOW_CONFUSION_MATRICES = True
SHOW_PER_ROOM_PLOTS = False

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
if preproc_opts.get("normalization") == "empty_baseline":
    preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)
feature_cache_dir = get_cache_path(preproc_opts, feat_opts)
results_dir = get_results_path()
print(f"Feature cache path: {feature_cache_dir}")
print(f"Results path: {results_dir}")
tables_dir = results_dir / "tables"
plots_dir = results_dir / "plots"
for directory in (tables_dir, plots_dir):
    directory.mkdir(parents=True, exist_ok=True)


def _slugify(value: str) -> str:
    """Convert a display value to a compact filename-safe slug."""
    return value.lower().replace(".", "-").replace(" ", "-").strip("-")

Feature cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0
Results path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results


## Raw Data

In [3]:
magnitude_data, csv_diagnostics = load_raw_csi_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
)

Scenarios present: 1
Locations: 53 | Users: 6 | ESPs: 19


In [4]:
if SHOW_MAGNITUDE_PLOT:
    processed_magnitude_data, _ = process_magnitude_data(magnitude_data, **preproc_opts)
    print("=== RAW magnitudes (before normalization) ===")
    plot_magnitude_analysis_interactive(magnitude_data)
    print(
        "=== NORMALIZED magnitudes (after normalization: "
        f"{preproc_opts.get('normalization', 'none')}) ==="
    )
    plot_magnitude_analysis_interactive(processed_magnitude_data)
    del processed_magnitude_data

#### Feature Dataframes

In [5]:
feature_dataframes = load_feature_dataframes(
    magnitude_data,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    bands_to_run=BANDS_TO_RUN,
)

none_reference_dataframes = (
    feature_dataframes
    if preproc_opts.get("normalization") == "none"
    else load_or_build_none_reference_features(
        magnitude_data,
        active_preproc_opts=preproc_opts,
        feat_opts=feat_opts,
    )
)
fisher_diagnostics = print_normalization_discriminability(
    feature_dataframes,
    normalization=preproc_opts.get("normalization", "none"),
    reference_feature_dataframes=none_reference_dataframes,
    bands_to_run=BANDS_TO_RUN,
)
display(fisher_diagnostics)
del magnitude_data


[features] resolved cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0
[cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0/2_4ghz.parquet
[cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0/5ghz.parquet
[cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0/fusion.parquet
2.4 GHz: 13588 windows, 2708 columns
5 GHz: 14478 windows, 3368 columns
Fusion: 13568 windows, 6068 columns
[normalization diagnostic] none reference cache: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-none/feat=win60-step0
[cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc

,dataset,normalization,median_fisher_ratio,none_median_fisher_ratio
0,2.4 GHz,empty_baseline,0.035751,0.028427
1,5 GHz,empty_baseline,0.070829,0.030535
2,Fusion,empty_baseline,0.050150,0.029641


#### Model Parameters

In [6]:
params_lookup = {}
if RUN_GRID_SEARCH:
    print("Parameter lookup deferred until the requested grid search completes.")
else:
    params_lookup = load_params_lookup(
        results_dir,
        models_to_run=MODELS_TO_RUN,
        bands_to_run=BANDS_TO_RUN,
        preproc_opts=preproc_opts,
        feat_opts=feat_opts,
        require_tuned_params=REQUIRE_TUNED_PARAMS,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_blocks=BLOCK_COUNT,
    )
    for key, value in params_lookup.items():
        print(f"{key}: {value}")

[tuned params] exact runs.csv match: RF / 2.4 GHz / run_id=ml__rf__2_4ghz__grid_search__ebl-session__s42__d8d534
[tuned params] exact runs.csv match: RF / 5 GHz / run_id=ml__rf__5ghz__grid_search__ebl-session__s42__0249b6
[tuned params] exact runs.csv match: RF / Fusion / run_id=ml__rf__fusion__grid_search__ebl-session__s42__666a25
[tuned params] exact runs.csv match: KNN / 2.4 GHz / run_id=ml__knn__2_4ghz__grid_search__ebl-session__s42__d47768
[tuned params] exact runs.csv match: KNN / 5 GHz / run_id=ml__knn__5ghz__grid_search__ebl-session__s42__80bca5
[tuned params] exact runs.csv match: KNN / Fusion / run_id=ml__knn__fusion__grid_search__ebl-session__s42__5b6367
[tuned params] exact runs.csv match: SVM / 2.4 GHz / run_id=ml__svm__2_4ghz__grid_search__ebl-session__s42__3be980
[tuned params] exact runs.csv match: SVM / 5 GHz / run_id=ml__svm__5ghz__grid_search__ebl-session__s42__0f2dc1
[tuned params] exact runs.csv match: SVM / Fusion / run_id=ml__svm__fusion__grid_search__ebl-session

##### Optional Grid Search

In [7]:
grid_ran = run_optional_grid_search(
    feature_dataframes,
    run_grid_search=RUN_GRID_SEARCH,
    results_dir=results_dir,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    n_blocks=BLOCK_COUNT,
    row_spacing=ROW_SPACING,
    column_spacing=COLUMN_SPACING,
)
if grid_ran:
    raise SystemExit("RUN_GRID_SEARCH=True completed; set it to False before running experiments.")

#### Global 52-Class Baselines

In [8]:
global_summary, global_predictions_by_key = run_global_baselines(
    feature_dataframes,
    params_lookup=params_lookup,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    split_modes=SPLIT_MODES,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    n_blocks=BLOCK_COUNT,
    n_jobs=N_JOBS,
    results_dir=results_dir,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    force_retrain=FORCE_RETRAIN,
    save_predictions=SAVE_PREDICTIONS,
    svm_fallback_seconds=SVM_FUSION_FALLBACK_SECONDS,
    row_spacing=ROW_SPACING,
    column_spacing=COLUMN_SPACING,
)
display(global_summary)
run_registry = pd.read_csv(results_dir / "runs.csv")
active_run_ids = run_registry.loc[
    run_registry["model"].isin([model.lower() for model in MODELS_TO_RUN])
    & run_registry["split"].isin(SPLIT_MODES),
    "run_id",
]
if active_run_ids.empty:
    raise RuntimeError("No active run_id is available for plot storage.")
plots_dir = results_dir / "plots" / active_run_ids.iloc[0]
plots_dir.mkdir(parents=True, exist_ok=True)


[trial filter] split=lovo trials=['01'] kept=8906/13588
[protocol] split=lovo fold=01 trials_used=['01'] n_train=7336 n_test=1570 users=['01', '02', '03', '04', '05', '06'] train_users=['02', '03', '04', '05', '06'] test_users=['01']
[protocol] split=lovo fold=02 trials_used=['01'] n_train=7492 n_test=1414 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '03', '04', '05', '06'] test_users=['02']
[protocol] split=lovo fold=03 trials_used=['01'] n_train=7319 n_test=1587 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '02', '04', '05', '06'] test_users=['03']
[protocol] split=lovo fold=04 trials_used=['01'] n_train=7346 n_test=1560 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '02', '03', '05', '06'] test_users=['04']
[protocol] split=lovo fold=05 trials_used=['01'] n_train=7420 n_test=1486 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '02', '03', '04', '06'] test_users=['05']
[protocol] split=lovo fold=06 trials_used=['01'] n_tra

/home/pedro.monteiro@co.it.pt/miniconda3/envs/thesis/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/ml__svm__2_4ghz__lovo__ebl-session__s42__a7141d__fold-01.parquet
[LOVO] fold 1/6 done: acc=0.1000 fit=28.7s predict=29.6s wall=58.3s
[LOVO] 2.4 GHz / SVM fold 2/6: held_out_user=02


/home/pedro.monteiro@co.it.pt/miniconda3/envs/thesis/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/ml__svm__2_4ghz__lovo__ebl-session__s42__a7141d__fold-02.parquet
[LOVO] fold 2/6 done: acc=0.0714 fit=26.8s predict=25.5s wall=52.4s
[LOVO] 2.4 GHz / SVM fold 3/6: held_out_user=03


...

/home/pedro.monteiro@co.it.pt/miniconda3/envs/thesis/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/ml__svm__fusion__lovo__ebl-session__s42__a7141d__fold-05.parquet
[LOVO] fold 5/6 done: acc=0.2073 fit=147.2s predict=132.0s wall=279.2s
[LOVO] Fusion / SVM fold 6/6: held_out_user=06


/home/pedro.monteiro@co.it.pt/miniconda3/envs/thesis/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/ml__svm__fusion__lovo__ebl-session__s42__a7141d__fold-06.parquet
[LOVO] fold 6/6 done: acc=0.0753 fit=135.7s predict=98.0s wall=233.7s
[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/ml__svm__fusion__lovo__ebl-session__s42__a7141d.parquet
[LOVO] Fusion / SVM complete: position_accuracy=0.1402 +/- 0.0570, total_wall=1611.8s
[trial filter] split=lovo trials=['01'] kept=8889/13568
[protocol] split=lovo fold=01 trials_used=['01'] n_train=7331 n_test=1558 users=['01', '02', '03', '04', '05', '06'] train_users=['02', '03', '04', '05', '06'] test_users=['01']
[protocol] split=lovo fold=02 trials_used=['01'] n_train=7478 n_test=1411 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '03', '04', '05', '06'] test_users=['02']
[protocol] split=lovo fold=03 trials_used=['01'] n_train=7303 n_test=1586 users=['01', '02', '03', '04', '05', '06'] train_us

,dataset,model,split,n_estimators,max_features,max_depth,min_samples_split,min_samples_leaf,class_weight,position_accuracy_mean,...,fit_seconds,predict_seconds,wall_seconds,used_estimator,n_neighbors,weights,metric,kernel,C,gamma
0,2.4 GHz,RF,lovo,800.0,log2,40.0,2.0,1.0,balanced,0.134913,...,53.125445,1.316940,55.327176,RandomForestClassifier,NaN,NaN,NaN,NaN,NaN,NaN
1,2.4 GHz,KNN,lovo,NaN,NaN,NaN,NaN,NaN,NaN,0.052838,...,1.599964,5.392488,7.870483,KNeighborsClassifier,1.0,uniform,manhattan,NaN,NaN,NaN
2,2.4 GHz,SVM,lovo,NaN,NaN,NaN,NaN,NaN,NaN,0.097987,...,166.286876,148.764240,315.892410,SVC,NaN,NaN,NaN,rbf,10.0,0.0001
3,5 GHz,RF,lovo,800.0,sqrt,40.0,5.0,2.0,NaN,0.141025,...,238.277695,1.346541,240.539089,RandomForestClassifier,NaN,NaN,NaN,NaN,NaN,NaN
4,5 GHz,KNN,lovo,NaN,NaN,NaN,NaN,NaN,NaN,0.084748,...,1.624723,7.727357,10.166899,KNeighborsClassifier,21.0,distance,manhattan,NaN,NaN,NaN
5,5 GHz,SVM,lovo,NaN,NaN,NaN,NaN,NaN,NaN,0.132931,...,206.036415,180.484138,387.254187,SVC,NaN,NaN,NaN,rbf,100.0,0.0001
6,Fusion,RF,lovo,800.0,log2,40.0,5.0,2.0,balanced,0.180660,...,46.510229,1.351346,48.908678,RandomForestClassifier,NaN,NaN,NaN,NaN,NaN,NaN
7,Fusion,KNN,lovo,NaN,NaN,NaN,NaN,NaN,NaN,0.073605,...,3.473888,11.371232,15.655516,KNeighborsClassifier,5.0,distance,manhattan,NaN,NaN,NaN
8,Fusion,SVM,lovo,NaN,NaN,NaN,NaN,NaN,NaN,0.140158,...,861.359514,749.710962,1611.814655,SVC,NaN,NaN,NaN,rbf,10.0,0.0001


#### Analysis Tables

In [9]:
all_global_predictions = load_all_predictions(
    results_dir,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    split_modes=SPLIT_MODES,
)

master_table = master_results_table(
    all_global_predictions,
    summary_path=results_dir / "runs.csv",
)
per_room_table = per_room_position_accuracy_table(all_global_predictions)
save_analysis_tables(master_table, per_room_table, tables_dir=tables_dir)

display(master_table)
display(per_room_table)

[tables] per-room predictions remain analysis-only and are not a runs.csv view.


,model,dataset,split,position_accuracy,macro_f1,room_accuracy,mean_distance_error,median_distance_error,rmse_distance_error,p90_distance_error,...,rmse_distance_error_std,p90_distance_error_mean,p90_distance_error_std,position_accuracy_pooled,majority_position_accuracy,majority_room_accuracy,fit_seconds,predict_seconds,wall_seconds,used_estimator
0,KNN,2.4 GHz,lovo,0.0528 +/- 0.0228,0.0433 +/- 0.0250,0.4981 +/- 0.0993,5.4289 +/- 0.5480,4.8644 +/- 0.7781,6.4516 +/- 0.5386,10.7022 +/- 0.8697,...,0.538563,10.702168,0.869673,0.053447,0.020705,0.651420,1.599964,5.392488,7.870483,KNeighborsClassifier
1,RF,2.4 GHz,lovo,0.1349 +/- 0.0797,0.1056 +/- 0.0706,0.5722 +/- 0.1448,4.2222 +/- 0.5838,3.8511 +/- 0.8961,5.2197 +/- 0.4315,8.6356 +/- 1.3085,...,0.431526,8.635571,1.308545,0.137211,0.020705,0.651420,53.125445,1.316940,55.327176,RandomForestClassifier
2,SVM,2.4 GHz,lovo,0.0980 +/- 0.0392,0.0832 +/- 0.0409,0.5560 +/- 0.1316,4.7458 +/- 0.6360,4.1345 +/- 0.6217,5.8643 +/- 0.7377,9.9779 +/- 1.3591,...,0.737720,9.977920,1.359125,0.099371,0.020705,0.651420,166.286876,148.764240,315.892410,SVC
3,KNN,5 GHz,lovo,0.0847 +/- 0.0248,0.0708 +/- 0.0238,0.4394 +/- 0.1314,5.1522 +/- 0.8942,4.5461 +/- 1.1829,6.2411 +/- 0.9291,10.3862 +/- 1.3240,...,0.929071,10.386167,1.324023,0.084889,0.019599,0.655680,1.624723,7.727357,10.166899,KNeighborsClassifier
4,RF,5 GHz,lovo,0.1410 +/- 0.0713,0.1136 +/- 0.0574,0.6892 +/- 0.1657,3.5501 +/- 0.6178,3.2530 +/- 0.7080,4.3581 +/- 0.5822,7.2605 +/- 1.1582,...,0.582212,7.260546,1.158171,0.141207,0.019599,0.655680,238.277695,1.346541,240.539089,RandomForestClassifier
5,SVM,5 GHz,lovo,0.1329 +/- 0.0426,0.1078 +/- 0.0470,0.6851 +/- 0.1254,3.8277 +/- 0.6665,3.1878 +/- 0.6366,4.9005 +/- 0.8311,8.3086 +/- 1.6415,...,0.831124,8.308633,1.641504,0.133161,0.019599,0.655680,206.036415,180.484138,387.254187,SVC
6,KNN,Fusion,lovo,0.0736 +/- 0.0294,0.0631 +/- 0.0301,0.4025 +/- 0.1007,5.2161 +/- 0.7244,4.6780 +/- 0.8601,6.2905 +/- 0.7702,10.4426 +/- 1.2892,...,0.770239,10.442599,1.289173,0.073912,0.020085,0.652058,3.473888,11.371232,15.655516,KNeighborsClassifier
7,RF,Fusion,lovo,0.1807 +/- 0.1129,0.1443 +/- 0.0984,0.6966 +/- 0.1762,3.4941 +/- 0.7540,3.0433 +/- 1.1341,4.3758 +/- 0.5478,7.1896 +/- 0.8548,...,0.547780,7.189633,0.854756,0.181798,0.020085,0.652058,46.510229,1.351346,48.908678,RandomForestClassifier
8,SVM,Fusion,lovo,0.1402 +/- 0.0570,0.1183 +/- 0.0601,0.7089 +/- 0.0689,3.6483 +/- 0.3896,3.0425 +/- 0.6116,4.6255 +/- 0.5053,7.7956 +/- 1.3693,...,0.505301,7.795570,1.369273,0.142648,0.020085,0.652058,861.359514,749.710962,1611.814655,SVC


,model,dataset,split,true_room,samples,position_accuracy
0,KNN,2.4 GHz,lovo,1,5799,0.047939
1,KNN,2.4 GHz,lovo,2,1707,0.042179
2,KNN,2.4 GHz,lovo,3,1400,0.090000
3,RF,2.4 GHz,lovo,1,5799,0.120883
4,RF,2.4 GHz,lovo,2,1707,0.188635
5,RF,2.4 GHz,lovo,3,1400,0.142143
6,SVM,2.4 GHz,lovo,1,5799,0.088464
7,SVM,2.4 GHz,lovo,2,1707,0.101347
8,SVM,2.4 GHz,lovo,3,1400,0.142143
9,KNN,5 GHz,lovo,1,6357,0.051911


#### LOVO cross-user analysis


In [10]:
if "lovo" in SPLIT_MODES:
    lovo_per_fold, lovo_summary = load_lovo_summary_tables(results_dir)
    lovo_table = lovo_aggregated_analysis_table(lovo_summary)
    save_lovo_analysis_table(lovo_table, tables_dir=tables_dir)

    plot_lovo_fold_spread(
        lovo_per_fold,
        bands=BANDS_TO_RUN,
        model="RF",
        save_path=plots_dir / f"{_slugify('lovo rf fold spread')}.png",
    )
    plot_block_vs_lovo_position_accuracy(
        globals().get("global_summary", master_table),
        lovo_summary,
        bands=BANDS_TO_RUN,
        model="RF",
        save_path=plots_dir / f"{_slugify('block vs lovo rf position accuracy')}.png",
    )

    display(lovo_table)
    display(lovo_per_fold.loc[lovo_per_fold["model"] == "RF"])
else:
    print("LOVO analysis skipped because 'lovo' is not in SPLIT_MODES.")

KeyError: 'model'

#### Analysis Figures

In [ ]:
if SHOW_CDF_BY_BAND:
    for band in BANDS_TO_RUN:
        plot_localization_error_cdf_by_model(
            all_global_predictions,
            dataset=band,
            save_path=plots_dir / f"cdf_by_model_{_slugify(band)}.png",
        )

if SHOW_CDF_BY_MODEL:
    for model in MODELS_TO_RUN:
        model_predictions = all_global_predictions.loc[all_global_predictions["model"] == model]
        plot_band_error_cdf(
            model_predictions,
            model_label=model,
            split_modes=SPLIT_MODES,
            band_order=BANDS_TO_RUN,
            save_path=plots_dir,
        )

if SHOW_BOXPLOT:
    plot_model_band_error_boxplot(
        all_global_predictions,
        models=MODELS_TO_RUN,
        bands=BANDS_TO_RUN,
        save_path=plots_dir / "boxplot_model_band_distance_error.png",
    )

#### Confusion Matrix And Floor Plan

In [ ]:
confusion_model, confusion_predictions = best_confusion_predictions(
    all_global_predictions,
    master_table,
    dataset=CONFUSION_DATASET,
    model=CONFUSION_MODEL,
)
print(f"Confusion/floor-plan model: {confusion_model} on {CONFUSION_DATASET}")

if SHOW_FLOOR_PLAN:
    plot_floor_plan_heatmap(
        confusion_predictions,
        title=f"{CONFUSION_DATASET} / {confusion_model} localization heatmap",
        save_path=plots_dir / f"floor_plan_{_slugify(CONFUSION_DATASET)}_{_slugify(confusion_model)}.png",
    )

if SHOW_CONFUSION_MATRICES:
    plot_global_position_confusion_matrix(
        confusion_predictions,
        dataset=CONFUSION_DATASET,
        normalize="true",
        save_path=plots_dir / f"confusion_{_slugify(CONFUSION_DATASET)}_{_slugify(confusion_model)}.png",
    )
    if SHOW_PER_ROOM_PLOTS:
        room_plot_dir = plots_dir / f"confusion_by_room_{_slugify(CONFUSION_DATASET)}_{_slugify(confusion_model)}"
        plot_position_confusion_by_true_room(
            confusion_predictions,
            dataset=CONFUSION_DATASET,
            normalize="true",
            save_path=room_plot_dir,
        )